# Deep-Live-Cam trên Google Colab T4

Chạy các cell theo thứ tự. Cell cuối giữ server Gradio hoạt động và in public URL để mở trên Mac.

In [ ]:
from pathlib import Path
import os, subprocess

# Colab normally starts in /content, so bootstrap the GitHub checkout first.
project = Path.cwd()
if not (project / 'requirements-colab.txt').exists():
    # Use the raw URL below; do not include Markdown brackets or parentheses.
    project = Path('/content/DeepFake')
    if project.exists() and not (project / '.git').exists():
        project = Path('/content/DeepFake_repo')
    if not (project / '.git').exists():
        repo_url = 'https://github.com/HelloDuongNha/DeepFake.git'
        # For a private repo, add a Colab Secret named GITHUB_TOKEN.
        # A fine-grained token only needs read access to this repository.
        github_token = os.environ.get('GITHUB_TOKEN', '')
        try:
            from google.colab import userdata
            github_token = github_token or userdata.get('GITHUB_TOKEN') or ''
        except Exception:
            pass
        clone_url = repo_url
        if github_token:
            clone_url = repo_url.replace('https://', f'https://x-access-token:{github_token}@')
        clone = subprocess.run(
            ['git', 'clone', clone_url, str(project)],
            text=True, capture_output=True,
        )
        if clone.returncode != 0:
            raise RuntimeError(
                'git clone failed. Make the repository Public, or add a Colab Secret '
                'named GITHUB_TOKEN with repository read access.\n' + clone.stderr
            )
os.chdir(project)
print('Project directory:', Path.cwd())

!nvidia-smi
!pip -q uninstall -y onnxruntime
!pip -q install -r requirements-colab.txt

import onnxruntime as ort
print('ONNX Runtime providers:', ort.get_available_providers())
assert 'CUDAExecutionProvider' in ort.get_available_providers(), 'CUDA EP chưa sẵn sàng; hãy Runtime > Restart session rồi chạy lại cell này.'

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen

BASE = 'https://huggingface.co/hacksider/deep-live-cam/resolve/main/'
root = Path('models')
root.mkdir(exist_ok=True)
files = {
    'inswapper_128.onnx': ('inswapper_128.onnx', root / 'inswapper_128.onnx'),
    'GPEN-BFR-512.onnx': ('GPEN-BFR-512.onnx', root / 'GPEN-BFR-512.onnx'),
    'gfpgan-1024.onnx': ('gfpgan-1024.onnx', root / 'gfpgan-1024.onnx'),
    '1k3d68.onnx': ('buffalo_l/buffalo_l/1k3d68.onnx', root / 'buffalo_l' / '1k3d68.onnx'),
    '2d106det.onnx': ('buffalo_l/buffalo_l/2d106det.onnx', root / 'buffalo_l' / '2d106det.onnx'),
    'det_10g.onnx': ('buffalo_l/buffalo_l/det_10g.onnx', root / 'buffalo_l' / 'det_10g.onnx'),
    'genderage.onnx': ('buffalo_l/buffalo_l/genderage.onnx', root / 'buffalo_l' / 'genderage.onnx'),
    'w600k_r50.onnx': ('buffalo_l/buffalo_l/w600k_r50.onnx', root / 'buffalo_l' / 'w600k_r50.onnx'),
}
for label, (remote, destination) in files.items():
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 1_000_000:
        print(f'skip {label}: {destination}')
        continue
    url = BASE + remote + '?download=true'
    print(f'downloading {label} ...')
    request = Request(url, headers={'User-Agent': 'Deep-Live-Cam-Colab'})
    with urlopen(request, timeout=120) as response, destination.open('wb') as output:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            output.write(chunk)
print('Models ready:', sorted(str(p) for p in root.rglob('*.onnx')))

In [ ]:
%env GRADIO_SHARE=1
!python colab_server.py --share